教学演示：任务拆解与依赖排序

**本课三个重点**：
1. 把复杂用户 Query **拆解为子任务**（任务分解提示词 + Agent 团队能力）。
2. 根据 Agent 技能的 **输入/输出参数** 分析子任务依赖，**调整执行顺序**。
3. （可选）调用 LLM 跑通「分解 → 排序 → 最终执行计划」完整链路。
`

## 0. 环境

```bash
pip install aiohttp dashscope python-dotenv
```

若后面要真调模型，请在项目根目录 `.env` 中设置（或在下文代码里临时写死）：

- `LLM_MODEL_VENDOR`（如 `qwen` / `chinageoqwen` / `gpt`）
- `LLM_MODEL_NAME`
- `LLM_API_KEY`

Agent 卡片默认从 `pre_data/agentcards/{zh|en}/*.json` 加载；若目录不存在，本 notebook 会使用内联示例数据便于课堂演示。

In [28]:
# Windows 下避免 print 长中文/特殊符号时报编码错（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

## 1. 课堂示例 Query

可改成你自己的测试用例；`lang` 决定使用中文还是英文提示词。

In [29]:
query = "你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略"
lang = "zh"  # "zh" 或 "en"
exclude_agent: list[str] = []

## 2. LLM 配置与客户端

与脚本中 `LlmConfig` + `get_llm_client` 相同：按 vendor 选择 Qwen / ChinaGeoQwen / Azure OpenAI。

In [30]:
import asyncio
import json
import os
from pathlib import Path
from typing import Any

import aiohttp
import dashscope
from dashscope.aigc.generation import AioGeneration
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")


class LlmConfig:
    llm_model_vendor: str = os.getenv("LLM_MODEL_VENDOR", "qwen")
    llm_model_name: str = os.getenv("LLM_MODEL_NAME", "qwen3-235b-a22b-instruct-2507")
     #需要添加自己的api key
    llm_api_key: str = os.getenv("LLM_API_KEY", "")


class BaseLLMClient:
    def __init__(self) -> None:
        self.system_prompt: str = ""

    def set_system_prompt(self, system_prompt: str):
        self.system_prompt = system_prompt

    async def chat(self, prompt: str, response_format: str = None) -> str:
        raise NotImplementedError

    async def parse_json(self, raw_output: str) -> dict:
        return json.loads(raw_output)


class QwenClient(BaseLLMClient):
    def __init__(self, model_name: str, api_key: str):
        self.model_name = model_name
        dashscope.api_key = api_key

    async def chat(self, query: str, response_format: str = None) -> str:
        return await _request_qwen_api(
            query, dashscope.api_key, self.model_name, self.system_prompt, response_format
        )


async def _request_qwen_api(mprompt, api_key, model, sys_prompt="You are a helpful assistant.", response_format=None):
    dashscope.api_key = api_key
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": mprompt},
    ]
    kwargs = dict(api_key=dashscope.api_key, model=model, messages=messages, result_format="message")
    if response_format == {"type": "json_object"}:
        kwargs["response_format"] = response_format
    response = await AioGeneration.call(**kwargs)
    return response["output"]["choices"][0]["message"]["content"]


def get_llm_client(vendor: str, model: str, api_key: str) -> BaseLLMClient:
    if vendor.startswith("qwen"):
        return QwenClient(model, api_key)
    raise ValueError(f"本 notebook 演示默认 qwen；当前 vendor={vendor}")


llm_client = get_llm_client(LlmConfig.llm_model_vendor, LlmConfig.llm_model_name, LlmConfig.llm_api_key)
llm_client.set_system_prompt("You are a helpful assistant.")
print(f"vendor={LlmConfig.llm_model_vendor}, model={LlmConfig.llm_model_name}, api_key={'*' * 8 if LlmConfig.llm_api_key else '(空)'}")

vendor=qwen, model=qwen3-235b-a22b-instruct-2507, api_key=********


## 3. 加载 Agent 团队

任务分解需要 Agent **描述**；依赖分析需要每个技能的 **inputSchema / outputSchema**。

In [31]:
DEMO_AGENT_CARDS_ZH = [
    {
        "name": "WeatherAgent",
        "description": "查询指定城市、日期的天气预报。",
        "skills": [
            {
                "name": "get_weather",
                "inputSchema": ["city", "date"],
                "outputSchema": ["weather_summary", "temperature"],
            }
        ],
    },
    {
        "name": "AttractionAgent",
        "description": "根据城市和偏好推荐旅游景点。",
        "skills": [
            {
                "name": "recommend_attractions",
                "inputSchema": ["city", "preferences"],
                "outputSchema": ["attraction_list"],
            }
        ],
    },
    {
        "name": "ItineraryAgent",
        "description": "根据天数、天气和景点列表生成行程安排。",
        "skills": [
            {
                "name": "plan_itinerary",
                "inputSchema": ["city", "days", "weather_summary", "attraction_list"],
                "outputSchema": ["itinerary"],
            }
        ],
    },
]


async def load_agent_cards(lang: str) -> list[dict[str, Any]]:
    cards_dir = PROJECT_ROOT / "pre_data" / "agentcards" / lang
    if cards_dir.exists():
        cards = []
        for card_file in cards_dir.glob("*.json"):
            with open(card_file, encoding="utf-8") as f:
                cards.append(json.load(f))
        print(f"✓ 从 {cards_dir} 加载 {len(cards)} 张 Agent 卡片")
        return cards
    print("⚠ 未找到 pre_data/agentcards，使用内联 DEMO 数据")
    return DEMO_AGENT_CARDS_ZH


def build_agent_team_text(agent_cards: list[dict], exclude_agent: list[str]) -> str:
    team = ""
    for card in agent_cards:
        if card["name"] in exclude_agent:
            continue
        team += card["description"] + "\n    "
    return team


def build_agent_parameters_text(agent_cards: list[dict], exclude_agent: list[str]) -> str:
    text = ""
    for card in agent_cards:
        name = card.get("name", "")
        if name in exclude_agent:
            continue
        text += f"{name}\n"
        for skill in card.get("skills", []):
            text += f"\t{skill['name']}, inputSchema:{skill.get('inputSchema', [])}, outputSchema:{skill.get('outputSchema', [])}\n"
    return text


agent_cards = await load_agent_cards(lang)
agent_team = build_agent_team_text(agent_cards, exclude_agent)
agent_parameters = build_agent_parameters_text(agent_cards, exclude_agent)
print("--- Agent 团队（任务分解用）---")
print(agent_team[:800] + ("..." if len(agent_team) > 800 else ""))

⚠ 未找到 pre_data/agentcards，使用内联 DEMO 数据
--- Agent 团队（任务分解用）---
查询指定城市、日期的天气预报。
    根据城市和偏好推荐旅游景点。
    根据天数、天气和景点列表生成行程安排。
    


## 4. 第一步：任务分解提示词

占位符：`{}` × 3 → 依次为 **背景信息**、**Agent 团队**、**用户输入**。

In [32]:
prompt_TP_zh = """
##角色：你是一个高度熟练的任务拆解专家，专注于将用户输入拆解为粒度适当的子任务。
##任务：根据用户的输入，分析目标以及智能体团队中所有智能体的能力，通过将用户输入分解为智能体团体能力能够覆盖的信息明确的子任务，以帮助有效地实现用户的目标。
##背景信息（关于具体任务拆解所需要知道的相关信息及要求）：
    {}
##智能体团队（可用于完成子任务的所有智能体能力）：
    {}
##步骤：
    1. **理解目标**：根据所有用户输入确定用户想要实现的核心目标和结果。
    2. **考虑背景信息**：提供的背景信息中包含拆解任务所需要的额外的领域知识，可用作参考。
    3. **定义子任务**：根据目标的复杂程度将任务拆解为子任务。
      3.1 **任务拆解原则**：
        - 简单：每个子任务都是一个能够匹配智能体团队能力的单一且信息明确的祈使句指令。
        - 信息完整：每个子任务指令中必须包含明确的槽位信息，所需槽位信息从用户输入及目标中抽取。
        - 优化复杂度：根据任务复杂度调整子任务数量，使用最少子任务完成整体目标。
      3.2 **子任务格式**：不同子任务间使用#分割。
    4. **确认子任务能否完成**：判断拆解出的每个子任务是否在智能体团队能力范围内，如不在该子任务为'NULL'。
    5. **补充子任务信息**：针对每个子任务，提取该子任务的所有槽位，并根据目标补全所有槽位，合并成一句完整的祈使指令生成新的子任务并替换原有子任务，使子任务内容信息明确指令清晰。
##要求：
    1.需要根据完整的用户输入的语义（包含所有输入内容）进行分析；
    2.背景信息仅作为参考，只有当需要时才使用；
    3.子任务仅能依靠智能体团队的智能体完成，避免擅自对智能体团队能力进行扩展并创造无法完成的子任务。
    4.仅输出简洁且信息完整的整体目标内容以及槽位信息完整的任务拆解结果，不要输出分析过程及原因;
    5.子任务中必须包含明确完整的槽位信息，避免直接将智能体能力作为子任务输出；
    6.输出的子任务应为信息明确的完整指令，确保用户输入及目标中包含的所有信息都出现在对应在子任务中，避免槽位信息丢失或单独输出槽位信息作为子任务。
##输出格式：
    # 目标
        整体目标内容（不包含人称的祈使句式）
    # 任务拆解
     - 子任务1
     - 子任务2(如有)
     - ...
##用户输入：
    {}
"""

prompt_TP_en = """
## Role
You are a highly skilled task decomposition expert, focusing on breaking down user inputs into appropriately granular subtasks.
## Task
Based on the user's input, analyze the user's goal and the capabilities of all agents in the agent team. Decompose the user input into clearly defined subtasks that fall within the capabilities of the agent team, to effectively help achieve the user's goal.
## Background Information (relevant information and requirements needed for task decomposition)
    {}
## Agent Team (all agent capabilities available to complete subtasks)
    {}
## Steps
    1. **Understand the Goal**: Determine the core goal and desired outcome based on all user input.
    2. **Consider Background Information**: Use background information as reference when needed.
    3. **Define Subtasks**: Decompose the task into subtasks based on the complexity of the goal.
    4. **Verify Subtask Feasibility**: Mark infeasible subtasks as 'NULL'.
    5. **Supplement Subtask Information**: Complete placeholders into full imperative instructions.
## Output Format
    # Goal
        Overall goal content (imperative sentence without personal pronouns)
    # Subtasks
     - Task1
     - Task2(if necessary)
     - ...
## User Input
    {}
"""

decompose_template = prompt_TP_zh if lang == "zh" else prompt_TP_en
background_info = ""
decompose_prompt = decompose_template.format(background_info, agent_team, query)
print(f"任务分解 Prompt 字符数: {len(decompose_prompt)}")

任务分解 Prompt 字符数: 1225


## 5. 展示：拼好后的「任务分解」完整提示词

下面就是第一步要发给模型的 **user 消息**。

In [33]:
from IPython.display import Markdown, display

display(Markdown(f"**字符数：** {len(decompose_prompt)}"))
display(Markdown("```text\n" + decompose_prompt[:2500] + ("\n...(截断)..." if len(decompose_prompt) > 2500 else "") + "\n```"))

**字符数：** 1225

```text

##角色：你是一个高度熟练的任务拆解专家，专注于将用户输入拆解为粒度适当的子任务。
##任务：根据用户的输入，分析目标以及智能体团队中所有智能体的能力，通过将用户输入分解为智能体团体能力能够覆盖的信息明确的子任务，以帮助有效地实现用户的目标。
##背景信息（关于具体任务拆解所需要知道的相关信息及要求）：
    
##智能体团队（可用于完成子任务的所有智能体能力）：
    查询指定城市、日期的天气预报。
    根据城市和偏好推荐旅游景点。
    根据天数、天气和景点列表生成行程安排。
    
##步骤：
    1. **理解目标**：根据所有用户输入确定用户想要实现的核心目标和结果。
    2. **考虑背景信息**：提供的背景信息中包含拆解任务所需要的额外的领域知识，可用作参考。
    3. **定义子任务**：根据目标的复杂程度将任务拆解为子任务。
      3.1 **任务拆解原则**：
        - 简单：每个子任务都是一个能够匹配智能体团队能力的单一且信息明确的祈使句指令。
        - 信息完整：每个子任务指令中必须包含明确的槽位信息，所需槽位信息从用户输入及目标中抽取。
        - 优化复杂度：根据任务复杂度调整子任务数量，使用最少子任务完成整体目标。
      3.2 **子任务格式**：不同子任务间使用#分割。
    4. **确认子任务能否完成**：判断拆解出的每个子任务是否在智能体团队能力范围内，如不在该子任务为'NULL'。
    5. **补充子任务信息**：针对每个子任务，提取该子任务的所有槽位，并根据目标补全所有槽位，合并成一句完整的祈使指令生成新的子任务并替换原有子任务，使子任务内容信息明确指令清晰。
##要求：
    1.需要根据完整的用户输入的语义（包含所有输入内容）进行分析；
    2.背景信息仅作为参考，只有当需要时才使用；
    3.子任务仅能依靠智能体团队的智能体完成，避免擅自对智能体团队能力进行扩展并创造无法完成的子任务。
    4.仅输出简洁且信息完整的整体目标内容以及槽位信息完整的任务拆解结果，不要输出分析过程及原因;
    5.子任务中必须包含明确完整的槽位信息，避免直接将智能体能力作为子任务输出；
    6.输出的子任务应为信息明确的完整指令，确保用户输入及目标中包含的所有信息都出现在对应在子任务中，避免槽位信息丢失或单独输出槽位信息作为子任务。
##输出格式：
    # 目标
        整体目标内容（不包含人称的祈使句式）
    # 任务拆解
     - 子任务1
     - 子任务2(如有)
     - ...
##用户输入：
    你能帮我规划一个五一假期的多城市旅行吗？我还没想好行程顺序…… 大概是上海、苏州、杭州这几个地方？需要包含行程路线、酒店推荐、天气情况和美食攻略

```

## 6.（可选）调用 LLM 做任务分解

把 `RUN_LLM = True` 且配置好 API Key 后运行；否则使用下方 **模拟响应** 继续后续步骤。

In [34]:
RUN_LLM = bool(LlmConfig.llm_api_key)

if RUN_LLM:
    decompose_response = await llm_client.chat(decompose_prompt)
    print("✓ 已调用 LLM 完成任务分解")


display(Markdown("### 模型输出（任务分解）"))
display(Markdown("```text\n" + decompose_response + "\n```"))

✓ 已调用 LLM 完成任务分解


### 模型输出（任务分解）

```text
# 目标
规划包含上海、苏州、杭州的五一假期多城市旅行，涵盖行程路线、酒店推荐、天气情况和美食攻略

# 任务拆解
- 查询上海市在五一假期期间的天气预报
- 查询苏州市在五一假期期间的天气预报
- 查询杭州市在五一假期期间的天气预报
- 根据偏好推荐上海市的旅游景点
- 根据偏好推荐苏州市的旅游景点
- 根据偏好推荐杭州市的旅游景点
- 根据五一假期天数、三地天气情况及推荐景点列表生成包含上海、苏州、杭州的多城市行程安排
```

## 7. 解析任务分解结果

从 `# 目标` / `# 任务拆解` 段落提取 **totalGoal** 与 **subSteps** 列表。

In [36]:
def parse_decomposition_response(response: str, lang: str) -> dict:
    totalgoal_key = "# 目标" if lang == "zh" else "# Goal"
    substep_key = "# 任务拆解" if lang == "zh" else "# Subtasks"

    total_goal = ""
    sub_steps = []
    goal_lines = []
    reached_tasks = False

    for line in response.split("\n"):
        line = line.strip()
        if not reached_tasks:
            if line.startswith(substep_key):
                reached_tasks = True
                continue
            if line.startswith(totalgoal_key):
                continue
            if line:
                goal_lines.append(line)
        elif line.startswith("-"):
            task = line.replace("- ", "").strip()
            if task:
                sub_steps.append(task)

    total_goal = " ".join(goal_lines)
    if not sub_steps:
        sub_steps = ["NULL"]
    return {"totalGoal": total_goal, "subSteps": sub_steps}


decomposition = parse_decomposition_response(decompose_response, lang)
print(f"总体目标: {decomposition['totalGoal']}")
print(f"子任务 ({len(decomposition['subSteps'])}):")
for i, step in enumerate(decomposition['subSteps'], 1):
    print(f"  {i}. {step}")

总体目标: 规划包含上海、苏州、杭州的五一假期多城市旅行，涵盖行程路线、酒店推荐、天气情况和美食攻略
子任务 (7):
  1. 查询上海市在五一假期期间的天气预报
  2. 查询苏州市在五一假期期间的天气预报
  3. 查询杭州市在五一假期期间的天气预报
  4. 根据偏好推荐上海市的旅游景点
  5. 根据偏好推荐苏州市的旅游景点
  6. 根据偏好推荐杭州市的旅游景点
  7. 根据五一假期天数、三地天气情况及推荐景点列表生成包含上海、苏州、杭州的多城市行程安排


## 8. 第二步：依赖分析提示词

为每个子任务分配临时 ID（T1、T2…），把 **subtasks** 与 **agents 技能参数** 填入 user prompt；system prompt 规定 JSON 排序输出格式。

In [37]:
# 与 analyzer/gen_substeps_order_test.py 第二部分完全一致
dependency_system_prompt_zh = """
你是一个多智能体系统中的[任务依赖分析器], 负责根据子任务与候选 Agent 技能之间的输入输出参数依赖关系, 识别子任务之间的执行依赖后调整子任务的执行顺序

# 输入
1. subtasks: 已拆分的子任务列表, 每个子任务包含:
   - task_id（如 T1、T2 等）
   - task_description（子任务的自然语言描述）
2. agents: 候选 Agent 列表, 每个 Agent 提供若干技能, 每个技能包含:
   - skill_name
   - input_parameters
   - output_parameters

# 目标
你的目标是分析子任务之间的必要依赖关系后调整子任务的执行顺序
第一步：建立全局输入输出视图  
- 遍历所有子任务  
- 对每个子任务，判断最可能由哪个 Agent 的哪项技能来完成  
- 读取并理解该技能的 输入参数 和 输出参数  
- 记录每个子任务的 [输入] 和 [输出]  

第二步：识别必要的任务依赖  
- 在获得所有子任务的输入输出信息后，从全局视角分析依赖关系 
- 如果子任务 A 输入参数中要求的信息，能够从子任务B的输出参数中获取，则A依赖B， B要先于A执行 
- 不要引入不确定或非必要的依赖关系

第三步：调整执行顺序  
- 仅当存在明确且必要的依赖关系时，才调整子任务顺序, 对不存在依赖关系的子任务，保持其原有的相对顺序  
- 调整后的顺序必须满足所有必要的依赖约束  
- 不要进行多余的重排

# 输出
- 输出必须是合法的 JSON。
- 仅输出调整后的子任务执行顺序, 使用如下格式:
{
  "1": "task_id",
  "2": "task_id",
  ...
}
其中order是执行顺序，从1开始递增; task_id是输入部分的子任务id, 必须与输入中的子任务对应
"""

dependency_user_prompt_zh = """
subtasks:
{subtasks}

agents:
{agents}

请严格按照输出格式返回, 除此之外不要有其他输出
"""

dependency_system_prompt_en = """
You are a task dependency analyzer in a multi-agent system.
Your responsibility is:
To identify required execution dependencies between subtasks based on input/output parameter dependencies between subtasks and candidate agent skills, then adjust the execution order of subtasks.

# Input:
The input consists of two parts:
1. subtasks: a list of subtasks. Each subtask includes:
    - task_id (e.g., T1, T2, …)
    - task_description (natural language description)

2. agents: a list of candidate agents. Each agent provides skills. Each skill includes:
    - skill_name
    - input_parameters
    - output_parameters

# Goal：
To adjust the execution order of subtasks based on parameter dependencies, without introducing unnecessary ordering constraints.
Follow the analysis steps strictly:

Step 1: Build a global input-output view  
- Iterate over all subtasks  
- For each subtask, infer the most likely agent skill that performs it  
- Read the input parameters and produced output parameters of that skill  
- Record only the inputs and outputs for each subtask  

Step 2: Identify required dependencies  
- After collecting input/output information for all subtasks, analyze dependencies from a global perspective  
- If input information of subtask A can be provided or consolidated by output information of subtask B, then: Subtask A depends on subtask B, meaning B must be executed before A.
- Identify dependencies through parameter relationships

Step 3: Adjust execution order  
- Adjust the subtask order ONLY when required dependencies exist  
- Preserve the original relative order of subtasks that have no dependency relations  
- The final order must satisfy all required dependencies  
- Avoid unnecessary reordering

Output rules (STRICT):
- Output MUST be valid JSON
- Output ONLY the adjusted execution order, Use the following format:
{
  "1": "task_id",
  "2": "task_id",
  ...
}
where order denotes the execution sequence, starting from 1 and incrementing progressively; 
task_id refers to the ID of the subtask in the input section, and must correspond to the subtasks specified in the input.
"""

dependency_user_prompt_en = """
subtasks:
{subtasks}

agents:
{agents}

Please return strictly in accordance with the specified output format, with no other content included.
"""

sub_steps = decomposition["subSteps"]
id_to_task = {f"T{i + 1}": task for i, task in enumerate(sub_steps)}

dependency_system_prompt = dependency_system_prompt_zh if lang == "zh" else dependency_system_prompt_en
dependency_user_template = dependency_user_prompt_zh if lang == "zh" else dependency_user_prompt_en
dependency_user_prompt = dependency_user_template.format(
    subtasks=id_to_task,
    agents=agent_parameters,
)

print("子任务 ID 映射:")
for tid, desc in id_to_task.items():
    print(f"  {tid}: {desc}")
print(f"\n依赖分析 user prompt 字符数: {len(dependency_user_prompt)}")

子任务 ID 映射:
  T1: 查询上海市在五一假期期间的天气预报
  T2: 查询苏州市在五一假期期间的天气预报
  T3: 查询杭州市在五一假期期间的天气预报
  T4: 根据偏好推荐上海市的旅游景点
  T5: 根据偏好推荐苏州市的旅游景点
  T6: 根据偏好推荐杭州市的旅游景点
  T7: 根据五一假期天数、三地天气情况及推荐景点列表生成包含上海、苏州、杭州的多城市行程安排

依赖分析 user prompt 字符数: 595


## 9. 展示：依赖分析完整提示词

In [38]:
display(Markdown("### System Prompt"))
display(Markdown("```text\n" + dependency_system_prompt.strip() + "\n```"))
display(Markdown("### User Prompt"))
display(Markdown("```text\n" + dependency_user_prompt + "\n```"))

### System Prompt

```text
你是一个多智能体系统中的[任务依赖分析器], 负责根据子任务与候选 Agent 技能之间的输入输出参数依赖关系, 识别子任务之间的执行依赖后调整子任务的执行顺序

# 输入
1. subtasks: 已拆分的子任务列表, 每个子任务包含:
   - task_id（如 T1、T2 等）
   - task_description（子任务的自然语言描述）
2. agents: 候选 Agent 列表, 每个 Agent 提供若干技能, 每个技能包含:
   - skill_name
   - input_parameters
   - output_parameters

# 目标
你的目标是分析子任务之间的必要依赖关系后调整子任务的执行顺序
第一步：建立全局输入输出视图  
- 遍历所有子任务  
- 对每个子任务，判断最可能由哪个 Agent 的哪项技能来完成  
- 读取并理解该技能的 输入参数 和 输出参数  
- 记录每个子任务的 [输入] 和 [输出]  

第二步：识别必要的任务依赖  
- 在获得所有子任务的输入输出信息后，从全局视角分析依赖关系 
- 如果子任务 A 输入参数中要求的信息，能够从子任务B的输出参数中获取，则A依赖B， B要先于A执行 
- 不要引入不确定或非必要的依赖关系

第三步：调整执行顺序  
- 仅当存在明确且必要的依赖关系时，才调整子任务顺序, 对不存在依赖关系的子任务，保持其原有的相对顺序  
- 调整后的顺序必须满足所有必要的依赖约束  
- 不要进行多余的重排

# 输出
- 输出必须是合法的 JSON。
- 仅输出调整后的子任务执行顺序, 使用如下格式:
{
  "1": "task_id",
  "2": "task_id",
  ...
}
其中order是执行顺序，从1开始递增; task_id是输入部分的子任务id, 必须与输入中的子任务对应
```

### User Prompt

```text

subtasks:
{'T1': '查询上海市在五一假期期间的天气预报', 'T2': '查询苏州市在五一假期期间的天气预报', 'T3': '查询杭州市在五一假期期间的天气预报', 'T4': '根据偏好推荐上海市的旅游景点', 'T5': '根据偏好推荐苏州市的旅游景点', 'T6': '根据偏好推荐杭州市的旅游景点', 'T7': '根据五一假期天数、三地天气情况及推荐景点列表生成包含上海、苏州、杭州的多城市行程安排'}

agents:
WeatherAgent
	get_weather, inputSchema:['city', 'date'], outputSchema:['weather_summary', 'temperature']
AttractionAgent
	recommend_attractions, inputSchema:['city', 'preferences'], outputSchema:['attraction_list']
ItineraryAgent
	plan_itinerary, inputSchema:['city', 'days', 'weather_summary', 'attraction_list'], outputSchema:['itinerary']


请严格按照输出格式返回, 除此之外不要有其他输出

```

## 10.（可选）调用 LLM 做依赖排序

期望返回 JSON，例如 `{"1": "T2", "2": "T1", "3": "T3"}`。

In [39]:
# 无 API 时：按 T1→T2→… 保持原顺序（课堂可改成 {"1":"T2","2":"T1"} 等观察重排效果）
MOCK_ORDER = {str(i + 1): tid for i, tid in enumerate(id_to_task)}

if len(sub_steps) < 2:
    sorted_steps = sub_steps
    order = None
    print("⚠ 子任务少于 2 个，跳过排序")
elif RUN_LLM:
    llm_client.set_system_prompt(dependency_system_prompt)
    dep_response = await llm_client.chat(dependency_user_prompt, {"type": "json_object"})
    order = await llm_client.parse_json(dep_response)
    print("✓ 已调用 LLM 完成依赖排序")
    print(json.dumps(order, ensure_ascii=False, indent=2))
else:
    order = MOCK_ORDER
    print("⚠ 使用 MOCK 排序:", order)

if len(sub_steps) >= 2:
    sorted_steps = [id_to_task[tid] for tid in order.values() if tid in id_to_task]
else:
    sorted_steps = sub_steps

✓ 已调用 LLM 完成依赖排序
{
  "1": "T1",
  "2": "T2",
  "3": "T3",
  "4": "T4",
  "5": "T5",
  "6": "T6",
  "7": "T7"
}


## 11. 最终执行计划

In [40]:
result = {
    "total_goal": decomposition["totalGoal"],
    "original_steps": sub_steps,
    "sorted_steps": sorted_steps,
}

print("=" * 60)
print("📊 最终执行计划")
print("=" * 60)
print(f"\n总体目标: {result['total_goal']}")
print(f"\n原始子任务顺序:")
for i, step in enumerate(result["original_steps"], 1):
    print(f"  {i}. {step}")
print(f"\n优化后执行顺序:")
for i, step in enumerate(result["sorted_steps"], 1):
    orig_id = next(k for k, v in id_to_task.items() if v == step)
    print(f"  第{i}步 [{orig_id}]: {step}")

output_file = PROJECT_ROOT / "analyzer" / "demo_result_notebook.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f"\n💾 结果已保存: {output_file}")

📊 最终执行计划

总体目标: 规划包含上海、苏州、杭州的五一假期多城市旅行，涵盖行程路线、酒店推荐、天气情况和美食攻略

原始子任务顺序:
  1. 查询上海市在五一假期期间的天气预报
  2. 查询苏州市在五一假期期间的天气预报
  3. 查询杭州市在五一假期期间的天气预报
  4. 根据偏好推荐上海市的旅游景点
  5. 根据偏好推荐苏州市的旅游景点
  6. 根据偏好推荐杭州市的旅游景点
  7. 根据五一假期天数、三地天气情况及推荐景点列表生成包含上海、苏州、杭州的多城市行程安排

优化后执行顺序:
  第1步 [T1]: 查询上海市在五一假期期间的天气预报
  第2步 [T2]: 查询苏州市在五一假期期间的天气预报
  第3步 [T3]: 查询杭州市在五一假期期间的天气预报
  第4步 [T4]: 根据偏好推荐上海市的旅游景点
  第5步 [T5]: 根据偏好推荐苏州市的旅游景点
  第6步 [T6]: 根据偏好推荐杭州市的旅游景点
  第7步 [T7]: 根据五一假期天数、三地天气情况及推荐景点列表生成包含上海、苏州、杭州的多城市行程安排

💾 结果已保存: D:\myproject\enterprise_bench_langchain\analyzer\demo_result_notebook.json


## 12. 小结

- **任务分解**：用户 Query + Agent 能力 → 带槽位的子任务列表（`# 目标` / `# 任务拆解`）。
- **依赖排序**：子任务 ID + Agent 技能 I/O → LLM 输出 JSON 执行顺序 → 重排子任务。